# Interactive Machine Learning Algorithm Explorer — Advanced AI Algorithms

A practical companion to `ml_landscape_all.ipynb`. This notebook adds the 12 algorithms from the supplied reference image, with concise explanations, runnable Python examples, and notes about what is and is not a complete production implementation.

**Install:** `pip install numpy matplotlib scikit-learn torch`

YOLO examples use Ultralytics; install `pip install ultralytics` separately if you want to run the optional detector cell. Synthetic data and small models here are for learning, not production benchmarking.

In [ ]:
import math, random
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
random.seed(42); np.random.seed(42)
print('NumPy', np.__version__)

## 1. Proximal Policy Optimization (PPO)

An on-policy actor-critic algorithm. PPO clips the policy probability ratio to limit overly large policy updates. The snippet below demonstrates the clipped surrogate objective; a full agent also needs an environment, rollout buffer, value loss, advantage estimation, and optimizer loop.

In [ ]:
import torch
import torch.nn.functional as F
old_logp = torch.tensor([-0.8, -1.1, -0.4])
new_logp = torch.tensor([-0.7, -1.3, -0.2], requires_grad=True)
advantages = torch.tensor([1.0, -0.5, 0.8])
ratio = torch.exp(new_logp - old_logp)
eps = 0.2
unclipped = ratio * advantages
clipped = torch.clamp(ratio, 1-eps, 1+eps) * advantages
policy_loss = -torch.minimum(unclipped, clipped).mean()
policy_loss.backward()
print('PPO clipped policy loss:', float(policy_loss.detach()))

## 2. A* Search

A* selects the frontier node minimizing `f(n) = g(n) + h(n)`: known path cost plus estimated remaining cost. An admissible heuristic never overestimates the true remaining cost.

In [ ]:
import heapq
grid = ['S....', '.##..', '...#.', '.#..G']
start=(0,0); goal=(4,3)
def astar(grid, start, goal):
    h=lambda p: abs(p[0]-goal[0])+abs(p[1]-goal[1])
    q=[(h(start),0,start)]; came={start:None}; cost={start:0}
    while q:
        _,g,p=heapq.heappop(q)
        if p==goal: break
        for dx,dy in ((1,0),(-1,0),(0,1),(0,-1)):
            n=(p[0]+dx,p[1]+dy)
            if not (0<=n[0]<len(grid[0]) and 0<=n[1]<len(grid)): continue
            if grid[n[1]][n[0]]=='#': continue
            ng=g+1
            if ng<cost.get(n,float('inf')):
                cost[n]=ng; came[n]=p; heapq.heappush(q,(ng+h(n),ng,n))
    if goal not in came: return None
    path=[]; p=goal
    while p is not None: path.append(p); p=came[p]
    return path[::-1]
path=astar(grid,start,goal); print('Path:',path,'steps:',len(path)-1 if path else None)

## 3. Monte Carlo Tree Search (MCTS)

MCTS repeatedly selects a promising node, expands it, simulates an outcome, and backpropagates the reward. This compact single-player example searches a toy binary decision tree; game implementations need state transitions, legal actions, terminal checks, and a reward model.

In [ ]:
import math, random
class Node:
    def __init__(self, state, parent=None): self.state,self.parent,self.children,self.n,self.w=state,parent,[],0,0.0
root=Node(0)
def rollout(state): return 1.0 if random.random()<state/10 else 0.0
for _ in range(300):
    node=root
    while node.children: node=max(node.children,key=lambda c:c.w/(c.n or 1)+1.4*math.sqrt(math.log(node.n+1)/(c.n+1)))
    if node.state<10:
        node.children=[Node(node.state+1,node),Node(node.state+2,node)]
        node=random.choice(node.children)
    reward=rollout(node.state)
    while node: node.n+=1; node.w+=reward; node=node.parent
print([(c.state,round(c.w/c.n,3)) for c in root.children])

## 4. Recurrent Neural Networks and LSTMs

An LSTM uses input, forget, and output gates plus a cell state to carry information across sequence steps. This PyTorch example learns a simple next-step mapping from synthetic sequences.

In [ ]:
import torch
from torch import nn
torch.manual_seed(42)
X=torch.randn(64,12,1); y=X[:,-1,:]
class LSTMRegressor(nn.Module):
    def __init__(self):
        super().__init__(); self.rnn=nn.LSTM(1,16,batch_first=True); self.head=nn.Linear(16,1)
    def forward(self,x):
        out,_=self.rnn(x); return self.head(out[:,-1])
model=LSTMRegressor(); opt=torch.optim.Adam(model.parameters(),lr=0.01)
for _ in range(30):
    opt.zero_grad(); loss=nn.functional.mse_loss(model(X),y); loss.backward(); opt.step()
print('LSTM training loss:',round(loss.item(),4))

## 5. Differentiable Neural Computer (DNC)

A DNC combines a neural controller with an external memory matrix and differentiable read/write operations. The following is a minimal differentiable memory read/write illustration, not a complete DNC (which also needs allocation, temporal links, usage tracking, and addressing).

In [ ]:
import torch
torch.manual_seed(7)
memory=torch.zeros(4,3,requires_grad=True)
write_weights=torch.softmax(torch.randn(4),dim=0)
write_vector=torch.tensor([0.2,0.5,0.9])
updated=memory + write_weights[:,None]*write_vector[None,:]
read_weights=torch.softmax(torch.randn(4),dim=0)
read_vector=read_weights @ updated
read_vector.sum().backward()
print('Read vector:',read_vector.detach().numpy())
print('Memory gradient exists:',memory.grad is not None)

## 6. Alpha-Beta Pruning

Alpha-beta pruning accelerates minimax by discarding branches that cannot affect the final decision. The example assumes a two-player, zero-sum game and a supplied game tree.

In [ ]:
def alphabeta(node, depth, alpha, beta, maximizing):
    if depth==0 or isinstance(node,int): return node
    if maximizing:
        value=-float('inf')
        for child in node:
            value=max(value,alphabeta(child,depth-1,alpha,beta,False)); alpha=max(alpha,value)
            if alpha>=beta: break
        return value
    value=float('inf')
    for child in node:
        value=min(value,alphabeta(child,depth-1,alpha,beta,True)); beta=min(beta,value)
        if beta<=alpha: break
    return value
tree=[[3,5],[2,9]]
print('Best minimax value:',alphabeta(tree,2,-float('inf'),float('inf'),True))

## 7. YOLO object detection

YOLO predicts object boxes and classes in a single forward pass. This optional cell uses a pretrained Ultralytics model and downloads weights on first run. Only run it when you intentionally want that external model download; inference results depend on the model version and input image.

In [ ]:
# Optional: pip install ultralytics
# from ultralytics import YOLO
# detector = YOLO('yolo11n.pt')  # downloads pretrained weights if absent
# results = detector('https://ultralytics.com/images/bus.jpg')
# results[0].show()

## 8. Generative Adversarial Networks (GANs)

A GAN trains a generator to produce samples that fool a discriminator. This small PyTorch example performs one discriminator and generator update on synthetic 1-D real data.

In [ ]:
import torch
from torch import nn
torch.manual_seed(42)
G=nn.Sequential(nn.Linear(1,16),nn.ReLU(),nn.Linear(16,1))
D=nn.Sequential(nn.Linear(1,16),nn.LeakyReLU(0.2),nn.Linear(16,1),nn.Sigmoid())
go=torch.optim.Adam(G.parameters(),lr=0.01); do=torch.optim.Adam(D.parameters(),lr=0.01)
real=torch.randn(32,1)*0.5+2; noise=torch.randn(32,1); fake=G(noise).detach()
bce=nn.BCELoss(); do.zero_grad()
dl=bce(D(real),torch.ones(32,1))+bce(D(fake),torch.zeros(32,1)); dl.backward(); do.step()
go.zero_grad(); generated=G(torch.randn(32,1)); gl=bce(D(generated),torch.ones(32,1)); gl.backward(); go.step()
print('D loss:',round(dl.item(),4),'G loss:',round(gl.item(),4))

## 9. VGGNet and ResNet

VGG uses repeated small convolutions; ResNet adds skip connections to ease optimization of deeper networks. The following PyTorch snippet compares the core residual-block shape without implementing full ImageNet-scale models.

In [ ]:
import torch
from torch import nn
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(channels,channels,3,padding=1,bias=False),nn.BatchNorm2d(channels),nn.ReLU(),nn.Conv2d(channels,channels,3,padding=1,bias=False),nn.BatchNorm2d(channels))
    def forward(self,x): return torch.relu(x+self.net(x))
block=ResidualBlock(16); sample=torch.randn(2,16,32,32)
print('ResNet block output shape:',tuple(block(sample).shape))
# VGG-style stack: nn.Sequential(nn.Conv2d(3,64,3,padding=1), nn.ReLU(), nn.Conv2d(64,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))

## 10. Behavior Trees

Behavior trees compose conditions and actions hierarchically. A sequence succeeds only if every child succeeds; a selector succeeds when one child succeeds.

In [ ]:
SUCCESS,FAILURE='SUCCESS','FAILURE'
def sequence(*nodes):
    for node in nodes:
        result=node()
        if result!=SUCCESS: return result
    return SUCCESS
def selector(*nodes):
    for node in nodes:
        result=node()
        if result==SUCCESS: return SUCCESS
    return FAILURE
enemy_near=lambda: SUCCESS
health_ok=lambda: FAILURE
attack=lambda: SUCCESS
print('Behavior result:',selector(sequence(enemy_near,health_ok,attack),lambda:SUCCESS)())

## 11. Soft Actor-Critic (SAC)

SAC is an off-policy actor-critic algorithm for continuous control. It maximizes expected return plus policy entropy. This snippet demonstrates the entropy-regularized actor objective; it is not a complete SAC training loop (replay buffer, twin critics, target critics, and temperature tuning are omitted).

In [ ]:
import torch
torch.manual_seed(42)
log_prob=torch.tensor([-0.4,-1.2,-0.7],requires_grad=True)
q_value=torch.tensor([1.5,0.8,1.1])
alpha=0.2
actor_loss=(alpha*log_prob-q_value).mean()
actor_loss.backward()
print('SAC actor objective loss:',round(actor_loss.item(),4))

## 12. Attention: self-attention and cross-attention

Self-attention forms queries, keys, and values from the same sequence. Cross-attention uses queries from one sequence and keys/values from another (for example, decoder queries attending to encoder outputs).

In [ ]:
import torch, math
torch.manual_seed(42)
B,Tq,Tk,D=2,4,6,8
query=torch.randn(B,Tq,D); key=torch.randn(B,Tk,D); value=torch.randn(B,Tk,D)
scores=query @ key.transpose(-2,-1) / math.sqrt(D)
weights=torch.softmax(scores,dim=-1)
context=weights @ value
print('Cross-attention context shape:',tuple(context.shape))
# Self-attention: use x for all three: q=k=v=x, with a causal mask for autoregressive decoding.

## Algorithm selection summary

| Algorithm | Typical problem | Key consideration |
|---|---|---|
| PPO / SAC | Continuous-control reinforcement learning | Environment design, reward quality, sample efficiency |
| A* | Shortest-path search | Heuristic quality and admissibility |
| MCTS / alpha-beta | Game-tree search | Simulation cost; pruning assumptions |
| RNN / LSTM | Sequential data | Long-range dependencies and sequence length |
| DNC | Tasks requiring explicit external memory | More complex memory addressing and training |
| YOLO | Real-time object detection | Model size, latency, deployment hardware |
| GAN | Synthetic data generation | Training stability and mode collapse |
| VGG / ResNet | Image representation | Compute, depth, skip connections |
| Behavior trees | Hierarchical decision logic | Deterministic control and modular behavior |
| Attention | Sequence and multimodal relationships | Quadratic sequence cost for standard full attention |

**Production note:** these examples demonstrate core mechanics. Production systems need representative data, validation, monitoring, security controls, resource budgets, and task-specific evaluation.